In [1]:
import os
import MDAnalysis as mda
import prolif as plf
import pandas as pd
import seaborn as sns
import numpy as np
from matplotlib import pyplot as plt
from rdkit import DataStructs

# =============================================================================
# CONFIGURATION - Define all your systems here
# =============================================================================

systems = {
    "GGPP": {
        "root_dir": "/media/dlbox/6TB/OPENFOLD/run/",
        "replica_folders": ["MD", "MD2", "MD3"],
        "traj_filename": "MMPBSA/center2x.nc",
        "ligand_resnames": "UN*",  # Main ligand to analyze (excluding MG)
        "cofactor_resnames": "MG",  # Optional cofactor
        "description": "PSY with GGPP ligand"
    },
    "FSPP": {
        "root_dir": "/media/dlbox/6TB/OPENFOLD/with_FPP2/run",
        "replica_folders": ["MD", "MD2", "MD3"],
        "traj_filename": "MMPBSA/center2x.nc",
        "ligand_resnames": "LG*",  # Main ligand to analyze (excluding MG)
        "cofactor_resnames": "MG",  # Optional cofactor
        "description": "PSY with FSPP ligand"
    }
}

# Analysis parameters
frame_step = 50  # Analyze every Nth frame
distance_cutoff = 10.0  # Å around ligand for protein selection
interaction_threshold_low = 50.0  # % for filtering
interaction_threshold_high = 80.0  # % for high-frequency interactions

# Output directory
output_dir = "2ligand_comparison_analysis_PSY"
os.makedirs(output_dir, exist_ok=True)

# =============================================================================
# FUNCTIONS
# =============================================================================

def load_system(system_name, system_config):
    """Load all replicas for a given system."""
    print(f"\n{'='*70}")
    print(f"Loading System: {system_name} - {system_config['description']}")
    print(f"{'='*70}")
    
    universes = []
    root_dir = system_config['root_dir']
    
    for replica in system_config['replica_folders']:
        top_file = f"{root_dir}/{replica}/solvated.prmtop"
        traj_file = f"{root_dir}/{replica}/{system_config['traj_filename']}"
        
        if not os.path.exists(top_file):
            print(f"WARNING: Topology file not found: {top_file}")
            continue
        if not os.path.exists(traj_file):
            print(f"WARNING: Trajectory file not found: {traj_file}")
            continue
            
        print(f"  Loading {replica}: {traj_file}")
        try:
            u = mda.Universe(top_file, traj_file)
            universes.append({'universe': u, 'replica': replica})
            print(f"    ✓ Loaded {len(u.trajectory)} frames")
        except Exception as e:
            print(f"    ✗ Error loading {replica}: {e}")
            
    return universes

def create_selections(universes, system_config, distance_cutoff):
    """Create ligand and protein selections for all replicas."""
    selections = []
    
    ligand_resnames = system_config['ligand_resnames']
    cofactor_resnames = system_config.get('cofactor_resnames', '')
    
    for entry in universes:
        u = entry['universe']
        replica = entry['replica']
        
        try:
            # Select main ligand only
            ligand_sel = u.select_atoms(f"resname {ligand_resnames}")
            
            # Select all ligands including cofactor for display (optional)
            if cofactor_resnames:
                all_ligands = u.select_atoms(f"resname {ligand_resnames} {cofactor_resnames}")
            else:
                all_ligands = ligand_sel
            
            # Select protein within distance cutoff of main ligand
            protein_sel = u.select_atoms(
                f"protein and byres around {distance_cutoff} group ligand",
                ligand=ligand_sel
            )
            
            # Create ProLIF molecules
            ligand_mol = plf.Molecule.from_mda(ligand_sel)
            protein_mol = plf.Molecule.from_mda(protein_sel)
            
            selections.append({
                'replica': replica,
                'universe': u,
                'ligand_sel': ligand_sel,
                'protein_sel': protein_sel,
                'ligand_mol': ligand_mol,
                'protein_mol': protein_mol
            })
            
            print(f"  [{replica}] Ligand: {len(ligand_sel)} atoms, "
                  f"Protein: {len(protein_sel.residues)} residues")
            
        except Exception as e:
            print(f"  [{replica}] Error creating selections: {e}")
            
    return selections

def normalize_interaction_keys(df):
    """
    Normalize interaction keys to remove ligand residue information.
    ProLIF uses MultiIndex columns with levels: ligand, protein, interaction
    We need to drop the ligand level and group by (protein, interaction)
    """
    # Check if columns are MultiIndex
    if isinstance(df.columns, pd.MultiIndex):
        # Get the number of levels
        n_levels = df.columns.nlevels
        
        if n_levels == 3:
            # Format: (ligand, protein, interaction)
            # Drop ligand level (level 0), keep protein and interaction
            df_normalized = df.copy()
            
            # Create new column labels without ligand info
            new_cols = [(col[1], col[2]) for col in df.columns]
            df_normalized.columns = pd.MultiIndex.from_tuples(new_cols, names=['protein', 'interaction'])
            
            # Group by the new column names (sum if multiple ligands interact with same residue)
            df_normalized = df_normalized.T.groupby(level=[0, 1]).sum().T
            
        else:
            print(f"  Warning: Unexpected MultiIndex levels: {n_levels}")
            df_normalized = df
    else:
        # If not MultiIndex, try tuple-based normalization
        new_columns = {}
        for col in df.columns:
            if isinstance(col, tuple) and len(col) == 3:
                # col format: (ligand_res, protein_res, interaction_type)
                ligand_res, protein_res, interaction_type = col
                new_key = (protein_res, interaction_type)
                new_columns[col] = new_key
            else:
                new_columns[col] = col
        
        df_normalized = df.rename(columns=new_columns)
        
        # If duplicate columns exist, sum them
        if df_normalized.columns.duplicated().any():
            df_normalized = df_normalized.T.groupby(level=0).sum().T
    
    return df_normalized

def run_fingerprint_analysis(selections, system_name, frame_step, output_dir):
    """Run interaction fingerprint analysis."""
    print(f"\n{'='*70}")
    print(f"Running Fingerprint Analysis: {system_name}")
    print(f"{'='*70}")
    
    results = []
    
    for sel in selections:
        replica = sel['replica']
        print(f"\n  [{replica}] Analyzing every {frame_step}th frame...")
        
        try:
            # Initialize and run fingerprint
            fp = plf.Fingerprint()
            fp.run(
                sel['universe'].trajectory[0::frame_step],
                sel['ligand_sel'],
                sel['protein_sel']
            )
            
            # Save fingerprint
            pkl_file = os.path.join(output_dir, f"fp_{system_name}_{replica}.pkl")
            fp.to_pickle(pkl_file)
            
            # Convert to dataframe and normalize
            df = fp.to_dataframe()
            df_normalized = normalize_interaction_keys(df)
            
            results.append({
                'replica': replica,
                'fingerprint': fp,
                'dataframe': df,
                'dataframe_normalized': df_normalized,
                'pkl_file': pkl_file
            })
            
            print(f"    ✓ Analyzed {len(df)} frames")
            print(f"    ✓ Found {len(df_normalized.columns)} unique protein interactions")
            
        except Exception as e:
            print(f"    ✗ Error: {e}")
            import traceback
            traceback.print_exc()
            
    return results

def combine_replica_data(results, system_name):
    """Combine interaction data across replicas using normalized data."""
    if not results:
        return None, None, None
    
    # Calculate mean interactions per replica (as percentage) using normalized data
    interaction_means = []
    all_normalized_dfs = []
    
    for res in results:
        df_norm = res['dataframe_normalized']
        all_normalized_dfs.append(df_norm)
        mean_vals = df_norm.mean().sort_values(ascending=False) * 100
        interaction_means.append(mean_vals.to_frame(name=res['replica']).T)
    
    # Combine all replicas
    combined_data = pd.concat(interaction_means, ignore_index=False)
    
    # Calculate overall average across replicas
    overall_mean = combined_data.mean().sort_values(ascending=False)
    
    # Concatenate all normalized dataframes for full dataset
    all_frames = pd.concat(all_normalized_dfs, ignore_index=True)
    
    return combined_data, overall_mean, all_frames

# =============================================================================
# MAIN ANALYSIS PIPELINE
# =============================================================================

print(f"\n{'#'*70}")
print(f"#  Multi-Ligand Interaction Comparison Analysis")
print(f"#  Comparing: {', '.join(systems.keys())}")
print(f"{'#'*70}")

# Store all system data
all_system_data = {}

# Process each system
for system_name, system_config in systems.items():
    system_output = os.path.join(output_dir, system_name)
    os.makedirs(system_output, exist_ok=True)
    
    # Load trajectories
    universes = load_system(system_name, system_config)
    if not universes:
        print(f"ERROR: No valid trajectories loaded for {system_name}")
        continue
    
    # Create selections
    selections = create_selections(
        universes,
        system_config,
        distance_cutoff
    )
    if not selections:
        print(f"ERROR: No valid selections for {system_name}")
        continue
    
    # Run fingerprint analysis
    results = run_fingerprint_analysis(
        selections,
        system_name,
        frame_step,
        system_output
    )
    if not results:
        print(f"ERROR: No fingerprint results for {system_name}")
        continue
    
    # Combine replica data (using normalized interactions)
    combined_data, overall_mean, all_frames = combine_replica_data(results, system_name)
    
    # Store system data
    all_system_data[system_name] = {
        'selections': selections,
        'results': results,
        'combined_data': combined_data,
        'overall_mean': overall_mean,
        'all_frames': all_frames,
        'output_dir': system_output,
        'config': system_config
    }
    
    # Save system-specific data
    if combined_data is not None:
        combined_data.T.to_csv(
            os.path.join(system_output, f"{system_name}_all_interactions.csv")
        )
        overall_mean.to_csv(
            os.path.join(system_output, f"{system_name}_mean_interactions.csv")
        )

# =============================================================================
# COMPARATIVE ANALYSIS
# =============================================================================

print(f"\n{'='*70}")
print(f"Comparative Analysis: Ligand Effects on Protein Interactions")
print(f"{'='*70}")

if len(all_system_data) < 2:
    print("ERROR: Need at least 2 systems for comparison")
else:
    # Combine all system means (now comparable because normalized)
    comparison_data = pd.DataFrame({
        name: data['overall_mean']
        for name, data in all_system_data.items()
    })
    
    # Get union of all interactions
    all_interactions = comparison_data.index
    print(f"\nTotal unique protein interactions found: {len(all_interactions)}")
    
    # Fill NaN with 0 for missing interactions
    comparison_data = comparison_data.fillna(0)
    
    # Filter interactions present in at least one system above threshold
    filtered_comparison = comparison_data[
        (comparison_data > interaction_threshold_low).any(axis=1)
    ]
    
    # Save comparison data
    comparison_file = os.path.join(output_dir, "ligand_comparison_all.csv")
    filtered_comparison.to_csv(comparison_file)
    print(f"\n✓ Saved comparison: {comparison_file}")
    print(f"  Interactions > {interaction_threshold_low}%: {len(filtered_comparison)}")
    
    # High-frequency interactions (>50% in at least one system)
    high_freq_comparison = comparison_data[
        (comparison_data > interaction_threshold_high).any(axis=1)
    ]
    
    high_freq_file = os.path.join(output_dir, "ligand_comparison_high_freq.csv")
    high_freq_comparison.to_csv(high_freq_file)
    print(f"✓ Saved high-frequency comparison: {high_freq_file}")
    print(f"  Interactions > {interaction_threshold_high}%: {len(high_freq_comparison)}")
    
    # =============================================================================
    # INTERACTION CATEGORIZATION
    # =============================================================================
    
    system_names = list(all_system_data.keys())
    
    # Categorize interactions
    common_interactions = comparison_data[
        (comparison_data[system_names[0]] > interaction_threshold_low) & 
        (comparison_data[system_names[1]] > interaction_threshold_low)
    ]
    
    unique_to_system1 = comparison_data[
        (comparison_data[system_names[0]] > interaction_threshold_low) & 
        (comparison_data[system_names[1]] <= interaction_threshold_low)
    ]
    
    unique_to_system2 = comparison_data[
        (comparison_data[system_names[0]] <= interaction_threshold_low) & 
        (comparison_data[system_names[1]] > interaction_threshold_low)
    ]
    
    print(f"\n{'='*70}")
    print(f"INTERACTION CATEGORIZATION")
    print(f"{'='*70}")
    print(f"Common interactions (>{interaction_threshold_low}% in both): {len(common_interactions)}")
    print(f"Unique to {system_names[0]}: {len(unique_to_system1)}")
    print(f"Unique to {system_names[1]}: {len(unique_to_system2)}")
    
    # Save categorized interactions
    common_interactions.to_csv(os.path.join(output_dir, "common_interactions.csv"))
    unique_to_system1.to_csv(os.path.join(output_dir, f"unique_to_{system_names[0]}.csv"))
    unique_to_system2.to_csv(os.path.join(output_dir, f"unique_to_{system_names[1]}.csv"))
    
    # =============================================================================
    # VISUALIZATIONS
    # =============================================================================
    
    print(f"\n{'='*70}")
    print(f"Generating Comparative Visualizations")
    print(f"{'='*70}")
    
    # 1. Comprehensive Heatmap (all significant interactions)
    if len(filtered_comparison) > 0:
        fig, ax = plt.subplots(figsize=(10, max(8, len(filtered_comparison) * 0.25)), dpi=150)
        
        sns.heatmap(
            filtered_comparison,
            annot=True,
            fmt='.1f',
            cmap='RdYlGn',
            center=50,
            vmin=0,
            vmax=100,
            cbar_kws={'label': 'Interaction Frequency (%)'},
            ax=ax,
            linewidths=0.5
        )
        
        ax.set_title('Ligand-Protein Interaction Comparison\n(Residue-Interaction Type)', 
                     fontsize=14, fontweight='bold')
        ax.set_xlabel('Ligand System', fontsize=12)
        ax.set_ylabel('Protein Interaction', fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        
        heatmap_file = os.path.join(output_dir, "comparison_heatmap_all.png")
        plt.savefig(heatmap_file, dpi=150, bbox_inches='tight')
        print(f"✓ Saved comprehensive heatmap: {heatmap_file}")
        plt.close()
    
    # 2. High-frequency interactions heatmap
    if len(high_freq_comparison) > 0:
        fig, ax = plt.subplots(figsize=(10, max(6, len(high_freq_comparison) * 0.3)), dpi=150)
        
        sns.heatmap(
            high_freq_comparison,
            annot=True,
            fmt='.1f',
            cmap='RdYlGn',
            center=50,
            vmin=0,
            vmax=100,
            cbar_kws={'label': 'Interaction Frequency (%)'},
            ax=ax,
            linewidths=0.5
        )
        
        ax.set_title('High-Frequency Interactions (>50%)\nAcross Ligand Systems', 
                     fontsize=14, fontweight='bold')
        ax.set_xlabel('Ligand System', fontsize=12)
        ax.set_ylabel('Protein Interaction', fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        
        heatmap_file = os.path.join(output_dir, "comparison_heatmap_high_freq.png")
        plt.savefig(heatmap_file, dpi=150, bbox_inches='tight')
        print(f"✓ Saved high-frequency heatmap: {heatmap_file}")
        plt.close()
    
    # 3. Scatter plot: direct comparison
    if len(all_system_data) == 2:
        fig, ax = plt.subplots(figsize=(10, 10), dpi=150)
        
        x = comparison_data[system_names[0]]
        y = comparison_data[system_names[1]]
        
        # Color points by category
        colors = []
        for idx in comparison_data.index:
            if idx in common_interactions.index:
                colors.append('blue')
            elif idx in unique_to_system1.index:
                colors.append('red')
            elif idx in unique_to_system2.index:
                colors.append('green')
            else:
                colors.append('gray')
        
        ax.scatter(x, y, alpha=0.6, c=colors, s=50)
        
        # Add diagonal line
        max_val = max(x.max(), y.max())
        ax.plot([0, max_val], [0, max_val], 'k--', alpha=0.3, label='Equal frequency')
        
        # Add threshold lines
        ax.axhline(y=interaction_threshold_low, color='orange', linestyle=':', alpha=0.5)
        ax.axvline(x=interaction_threshold_low, color='orange', linestyle=':', alpha=0.5)
        
        ax.set_xlabel(f'{system_names[0]} Interaction Frequency (%)', fontsize=12)
        ax.set_ylabel(f'{system_names[1]} Interaction Frequency (%)', fontsize=12)
        ax.set_title('Direct Ligand Comparison: Interaction Frequencies', 
                     fontsize=14, fontweight='bold')
        ax.grid(alpha=0.3)
        
        # Add legend
        from matplotlib.patches import Patch
        legend_elements = [
            Patch(facecolor='blue', label=f'Common (n={len(common_interactions)})'),
            Patch(facecolor='red', label=f'Unique to {system_names[0]} (n={len(unique_to_system1)})'),
            Patch(facecolor='green', label=f'Unique to {system_names[1]} (n={len(unique_to_system2)})'),
            Patch(facecolor='gray', label='Below threshold')
        ]
        ax.legend(handles=legend_elements, loc='upper left')
        
        plt.tight_layout()
        scatter_file = os.path.join(output_dir, "scatter_comparison.png")
        plt.savefig(scatter_file, dpi=150, bbox_inches='tight')
        print(f"✓ Saved scatter plot: {scatter_file}")
        plt.close()
    
    # 4. Difference analysis (for 2 systems)
    if len(all_system_data) == 2:
        diff = (comparison_data[system_names[0]] - comparison_data[system_names[1]])
        diff = diff[abs(diff) > 10].sort_values(ascending=False)
        
        if len(diff) > 0:
            fig, ax = plt.subplots(figsize=(12, max(8, len(diff) * 0.25)), dpi=150)
            
            colors = ['#2ca02c' if x > 0 else '#d62728' for x in diff.values]
            diff.plot(kind='barh', ax=ax, color=colors, width=0.8)
            
            ax.set_xlabel(f'Difference: {system_names[0]} - {system_names[1]} (%)', fontsize=12)
            ax.set_ylabel('Protein Interaction', fontsize=12)
            ax.set_title(f'Interaction Frequency Differences\nGreen: Favored by {system_names[0]}, Red: Favored by {system_names[1]}', 
                        fontsize=14, fontweight='bold')
            ax.axvline(x=0, color='black', linestyle='-', linewidth=1.5)
            ax.grid(axis='x', alpha=0.3)
            plt.tight_layout()
            
            diff_file = os.path.join(output_dir, "interaction_differences.png")
            plt.savefig(diff_file, dpi=150, bbox_inches='tight')
            print(f"✓ Saved difference plot: {diff_file}")
            plt.close()
            
            # Save difference data
            diff.to_csv(os.path.join(output_dir, "interaction_differences.csv"))
    
    # 5. Bar plot for top interactions in each category
    fig, axes = plt.subplots(1, 3, figsize=(20, 10), dpi=150)
    
    categories = [
        (common_interactions, 'Common Interactions', 'blue', axes[0]),
        (unique_to_system1, f'Unique to {system_names[0]}', 'red', axes[1]),
        (unique_to_system2, f'Unique to {system_names[1]}', 'green', axes[2])
    ]
    
    for data, title, color, ax in categories:
        if len(data) > 0:
            top_data = data.head(15)
            top_data.plot(kind='barh', ax=ax, width=0.8)
            ax.set_xlabel('Interaction Frequency (%)', fontsize=16)
            ax.set_title(title, fontsize=18, fontweight='bold')
            ax.legend(fontsize=14)
            ax.grid(axis='x', alpha=0.3)
            # Increase tick label sizes
            ax.tick_params(axis='both', labelsize=14)
        else:
            ax.text(0.5, 0.5, 'No interactions', ha='center', va='center', 
                    transform=ax.transAxes, fontsize=16)
            ax.set_title(title, fontsize=18, fontweight='bold')
    
    plt.tight_layout()
    category_file = os.path.join(output_dir, "interaction_categories.png")
    plt.savefig(category_file, dpi=150, bbox_inches='tight')
    print(f"✓ Saved category plot: {category_file}")
    plt.close()
    
    # 6. Generate barcode plots for each system
    for system_name, data in all_system_data.items():
        print(f"\n  Generating barcodes for {system_name}...")
        for res in data['results']:
            replica = res['replica']
            fp = res['fingerprint']
            
            try:
                fig = fp.plot_barcode(dpi=150, figsize=(15, 10))
                barcode_file = os.path.join(
                    data['output_dir'],
                    f"barcode_{system_name}_{replica}.png"
                )
                plt.savefig(barcode_file, dpi=150, bbox_inches='tight')
                plt.close()
                print(f"    ✓ {replica} barcode saved")
            except Exception as e:
                print(f"    ✗ Error creating barcode for {replica}: {e}")
    
    # =============================================================================
    # SUMMARY REPORT
    # =============================================================================
    
    print(f"\n{'='*70}")
    print(f"SUMMARY REPORT: Ligand Comparison")
    print(f"{'='*70}")
    
    for system_name, data in all_system_data.items():
        ligand_name = data['config']['ligand_resnames']
        print(f"\n{system_name} (Ligand: {ligand_name}):")
        print(f"  Replicas analyzed: {len(data['results'])}")
        if data['overall_mean'] is not None:
            total_interactions = len(data['overall_mean'][data['overall_mean'] > interaction_threshold_low])
            print(f"  Total significant interactions (>{interaction_threshold_low}%): {total_interactions}")
            top_5 = data['overall_mean'].head(5)
            print(f"  Top 5 interactions:")
            for interaction, freq in top_5.items():
                print(f"    - {interaction}: {freq:.1f}%")
    
    print(f"\n{'='*70}")
    print(f"Comparative Summary:")
    print(f"{'='*70}")
    print(f"Common interactions: {len(common_interactions)}")
    print(f"Unique to {system_names[0]}: {len(unique_to_system1)}")
    print(f"Unique to {system_names[1]}: {len(unique_to_system2)}")
    
    print(f"\n{'='*70}")
    print(f"Analysis Complete!")
    print(f"{'='*70}")
    print(f"\nAll outputs saved to: {output_dir}/")
    print(f"\nKey files:")
    print(f"  - ligand_comparison_all.csv: All significant interactions")
    print(f"  - ligand_comparison_high_freq.csv: High-frequency interactions")
    print(f"  - common_interactions.csv: Shared by both ligands")
    print(f"  - unique_to_*.csv: Ligand-specific interactions")
    print(f"  - comparison_heatmap_*.png: Visual comparisons")
    print(f"  - scatter_comparison.png: Direct frequency comparison")
    print(f"  - interaction_differences.png: Difference plot")
    print(f"  - interaction_categories.png: Categorized interactions")

/home/dlbox/miniconda3/lib/python3.12/site-packages/MDAnalysis/lib/pkdtree.py:33: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.4.1)
  from scipy.spatial import cKDTree
/home/dlbox/miniconda3/lib/python3.12/site-packages/MDAnalysis/topology/tables.py:52: DeprecationWarning: Deprecated in version 2.8.0
MDAnalysis.topology.tables has been moved to MDAnalysis.guesser.tables. This import point will be removed in MDAnalysis version 3.0.0
  warnings.warn(wmsg, category=DeprecationWarning)



######################################################################
#  Multi-Ligand Interaction Comparison Analysis
#  Comparing: GGPP, FSPP
######################################################################

Loading System: GGPP - PSY with GGPP ligand
  Loading MD: /media/dlbox/6TB/OPENFOLD/run//MD/MMPBSA/center2x.nc
    ✓ Loaded 10000 frames
  Loading MD2: /media/dlbox/6TB/OPENFOLD/run//MD2/MMPBSA/center2x.nc
    ✓ Loaded 10000 frames
  Loading MD3: /media/dlbox/6TB/OPENFOLD/run//MD3/MMPBSA/center2x.nc
    ✓ Loaded 10000 frames
  [MD] Ligand: 124 atoms, Protein: 146 residues
  [MD2] Ligand: 124 atoms, Protein: 138 residues
  [MD3] Ligand: 124 atoms, Protein: 143 residues

Running Fingerprint Analysis: GGPP

  [MD] Analyzing every 50th frame...


  0%|          | 0/200 [00:00<?, ?it/s]

    ✓ Analyzed 200 frames
    ✓ Found 83 unique protein interactions

  [MD2] Analyzing every 50th frame...


  0%|          | 0/200 [00:00<?, ?it/s]

...

    ✓ Analyzed 200 frames
    ✓ Found 66 unique protein interactions

  [MD2] Analyzing every 50th frame...


  0%|          | 0/200 [00:00<?, ?it/s]

    ✓ Analyzed 200 frames
    ✓ Found 67 unique protein interactions

  [MD3] Analyzing every 50th frame...


  0%|          | 0/200 [00:00<?, ?it/s]

    ✓ Analyzed 200 frames
    ✓ Found 65 unique protein interactions

Comparative Analysis: Ligand Effects on Protein Interactions

Total unique protein interactions found: 103

✓ Saved comparison: 2ligand_comparison_analysis_PSY/ligand_comparison_all.csv
  Interactions > 50.0%: 53
✓ Saved high-frequency comparison: 2ligand_comparison_analysis_PSY/ligand_comparison_high_freq.csv
  Interactions > 80.0%: 32

INTERACTION CATEGORIZATION
Common interactions (>50.0% in both): 33
Unique to GGPP: 9
Unique to FSPP: 11

Generating Comparative Visualizations
✓ Saved comprehensive heatmap: 2ligand_comparison_analysis_PSY/comparison_heatmap_all.png
✓ Saved high-frequency heatmap: 2ligand_comparison_analysis_PSY/comparison_heatmap_high_freq.png
✓ Saved scatter plot: 2ligand_comparison_analysis_PSY/scatter_comparison.png
✓ Saved difference plot: 2ligand_comparison_analysis_PSY/interaction_differences.png
✓ Saved category plot: 2ligand_comparison_analysis_PSY/interaction_categories.png

  Generating b

In [2]:
import os
import MDAnalysis as mda
import prolif as plf
import pandas as pd
import seaborn as sns
import numpy as np
from matplotlib import pyplot as plt
from rdkit import DataStructs

# =============================================================================
# CONFIGURATION - Define all your systems here
# =============================================================================

systems = {
    "GGPP": {
        "root_dir": "/media/dlbox/8TB/New_simulation/PQR/3w7f/with_GGPP/run/",
        "replica_folders": ["MD", "MD2", "MD3"],
        "traj_filename": "MMPBSA/center2x.nc",
        "ligand_resnames": "lg*",  # Main ligand to analyze (excluding MG)
        "cofactor_resnames": "MG",  # Optional cofactor
        "description": "PSY with GGPP ligand"
    },
    "FSPP": {
        "root_dir": "/media/dlbox/8TB/New_simulation/PQR/3w7f/run/",
        "replica_folders": ["MD", "MD2", "MD3"],
        "traj_filename": "MMPBSA/center2x.nc",
        "ligand_resnames": "IG*",  # Main ligand to analyze (excluding MG)
        "cofactor_resnames": "MG",  # Optional cofactor
        "description": "PSY with FSPP ligand"
    }
}

# Analysis parameters
frame_step = 50  # Analyze every Nth frame
distance_cutoff = 10.0  # Å around ligand for protein selection
interaction_threshold_low = 50.0  # % for filtering
interaction_threshold_high = 80.0  # % for high-frequency interactions

# Output directory
output_dir = "2ligand_comparison_analysis_CrtM"
os.makedirs(output_dir, exist_ok=True)

# =============================================================================
# FUNCTIONS
# =============================================================================

def load_system(system_name, system_config):
    """Load all replicas for a given system."""
    print(f"\n{'='*70}")
    print(f"Loading System: {system_name} - {system_config['description']}")
    print(f"{'='*70}")
    
    universes = []
    root_dir = system_config['root_dir']
    
    for replica in system_config['replica_folders']:
        top_file = f"{root_dir}/{replica}/solvated.prmtop"
        traj_file = f"{root_dir}/{replica}/{system_config['traj_filename']}"
        
        if not os.path.exists(top_file):
            print(f"WARNING: Topology file not found: {top_file}")
            continue
        if not os.path.exists(traj_file):
            print(f"WARNING: Trajectory file not found: {traj_file}")
            continue
            
        print(f"  Loading {replica}: {traj_file}")
        try:
            u = mda.Universe(top_file, traj_file)
            universes.append({'universe': u, 'replica': replica})
            print(f"    ✓ Loaded {len(u.trajectory)} frames")
        except Exception as e:
            print(f"    ✗ Error loading {replica}: {e}")
            
    return universes

def create_selections(universes, system_config, distance_cutoff):
    """Create ligand and protein selections for all replicas."""
    selections = []
    
    ligand_resnames = system_config['ligand_resnames']
    cofactor_resnames = system_config.get('cofactor_resnames', '')
    
    for entry in universes:
        u = entry['universe']
        replica = entry['replica']
        
        try:
            # Select main ligand only
            ligand_sel = u.select_atoms(f"resname {ligand_resnames}")
            
            # Select all ligands including cofactor for display (optional)
            if cofactor_resnames:
                all_ligands = u.select_atoms(f"resname {ligand_resnames} {cofactor_resnames}")
            else:
                all_ligands = ligand_sel
            
            # Select protein within distance cutoff of main ligand
            protein_sel = u.select_atoms(
                f"protein and byres around {distance_cutoff} group ligand",
                ligand=ligand_sel
            )
            
            # Create ProLIF molecules
            ligand_mol = plf.Molecule.from_mda(ligand_sel)
            protein_mol = plf.Molecule.from_mda(protein_sel)
            
            selections.append({
                'replica': replica,
                'universe': u,
                'ligand_sel': ligand_sel,
                'protein_sel': protein_sel,
                'ligand_mol': ligand_mol,
                'protein_mol': protein_mol
            })
            
            print(f"  [{replica}] Ligand: {len(ligand_sel)} atoms, "
                  f"Protein: {len(protein_sel.residues)} residues")
            
        except Exception as e:
            print(f"  [{replica}] Error creating selections: {e}")
            
    return selections

def normalize_interaction_keys(df):
    """
    Normalize interaction keys to remove ligand residue information.
    ProLIF uses MultiIndex columns with levels: ligand, protein, interaction
    We need to drop the ligand level and group by (protein, interaction)
    """
    # Check if columns are MultiIndex
    if isinstance(df.columns, pd.MultiIndex):
        # Get the number of levels
        n_levels = df.columns.nlevels
        
        if n_levels == 3:
            # Format: (ligand, protein, interaction)
            # Drop ligand level (level 0), keep protein and interaction
            df_normalized = df.copy()
            
            # Create new column labels without ligand info
            new_cols = [(col[1], col[2]) for col in df.columns]
            df_normalized.columns = pd.MultiIndex.from_tuples(new_cols, names=['protein', 'interaction'])
            
            # Group by the new column names (sum if multiple ligands interact with same residue)
            df_normalized = df_normalized.T.groupby(level=[0, 1]).sum().T
            
        else:
            print(f"  Warning: Unexpected MultiIndex levels: {n_levels}")
            df_normalized = df
    else:
        # If not MultiIndex, try tuple-based normalization
        new_columns = {}
        for col in df.columns:
            if isinstance(col, tuple) and len(col) == 3:
                # col format: (ligand_res, protein_res, interaction_type)
                ligand_res, protein_res, interaction_type = col
                new_key = (protein_res, interaction_type)
                new_columns[col] = new_key
            else:
                new_columns[col] = col
        
        df_normalized = df.rename(columns=new_columns)
        
        # If duplicate columns exist, sum them
        if df_normalized.columns.duplicated().any():
            df_normalized = df_normalized.T.groupby(level=0).sum().T
    
    return df_normalized

def run_fingerprint_analysis(selections, system_name, frame_step, output_dir):
    """Run interaction fingerprint analysis."""
    print(f"\n{'='*70}")
    print(f"Running Fingerprint Analysis: {system_name}")
    print(f"{'='*70}")
    
    results = []
    
    for sel in selections:
        replica = sel['replica']
        print(f"\n  [{replica}] Analyzing every {frame_step}th frame...")
        
        try:
            # Initialize and run fingerprint
            fp = plf.Fingerprint()
            fp.run(
                sel['universe'].trajectory[0::frame_step],
                sel['ligand_sel'],
                sel['protein_sel']
            )
            
            # Save fingerprint
            pkl_file = os.path.join(output_dir, f"fp_{system_name}_{replica}.pkl")
            fp.to_pickle(pkl_file)
            
            # Convert to dataframe and normalize
            df = fp.to_dataframe()
            df_normalized = normalize_interaction_keys(df)
            
            results.append({
                'replica': replica,
                'fingerprint': fp,
                'dataframe': df,
                'dataframe_normalized': df_normalized,
                'pkl_file': pkl_file
            })
            
            print(f"    ✓ Analyzed {len(df)} frames")
            print(f"    ✓ Found {len(df_normalized.columns)} unique protein interactions")
            
        except Exception as e:
            print(f"    ✗ Error: {e}")
            import traceback
            traceback.print_exc()
            
    return results

def combine_replica_data(results, system_name):
    """Combine interaction data across replicas using normalized data."""
    if not results:
        return None, None, None
    
    # Calculate mean interactions per replica (as percentage) using normalized data
    interaction_means = []
    all_normalized_dfs = []
    
    for res in results:
        df_norm = res['dataframe_normalized']
        all_normalized_dfs.append(df_norm)
        mean_vals = df_norm.mean().sort_values(ascending=False) * 100
        interaction_means.append(mean_vals.to_frame(name=res['replica']).T)
    
    # Combine all replicas
    combined_data = pd.concat(interaction_means, ignore_index=False)
    
    # Calculate overall average across replicas
    overall_mean = combined_data.mean().sort_values(ascending=False)
    
    # Concatenate all normalized dataframes for full dataset
    all_frames = pd.concat(all_normalized_dfs, ignore_index=True)
    
    return combined_data, overall_mean, all_frames

# =============================================================================
# MAIN ANALYSIS PIPELINE
# =============================================================================

print(f"\n{'#'*70}")
print(f"#  Multi-Ligand Interaction Comparison Analysis")
print(f"#  Comparing: {', '.join(systems.keys())}")
print(f"{'#'*70}")

# Store all system data
all_system_data = {}

# Process each system
for system_name, system_config in systems.items():
    system_output = os.path.join(output_dir, system_name)
    os.makedirs(system_output, exist_ok=True)
    
    # Load trajectories
    universes = load_system(system_name, system_config)
    if not universes:
        print(f"ERROR: No valid trajectories loaded for {system_name}")
        continue
    
    # Create selections
    selections = create_selections(
        universes,
        system_config,
        distance_cutoff
    )
    if not selections:
        print(f"ERROR: No valid selections for {system_name}")
        continue
    
    # Run fingerprint analysis
    results = run_fingerprint_analysis(
        selections,
        system_name,
        frame_step,
        system_output
    )
    if not results:
        print(f"ERROR: No fingerprint results for {system_name}")
        continue
    
    # Combine replica data (using normalized interactions)
    combined_data, overall_mean, all_frames = combine_replica_data(results, system_name)
    
    # Store system data
    all_system_data[system_name] = {
        'selections': selections,
        'results': results,
        'combined_data': combined_data,
        'overall_mean': overall_mean,
        'all_frames': all_frames,
        'output_dir': system_output,
        'config': system_config
    }
    
    # Save system-specific data
    if combined_data is not None:
        combined_data.T.to_csv(
            os.path.join(system_output, f"{system_name}_all_interactions.csv")
        )
        overall_mean.to_csv(
            os.path.join(system_output, f"{system_name}_mean_interactions.csv")
        )

# =============================================================================
# COMPARATIVE ANALYSIS
# =============================================================================

print(f"\n{'='*70}")
print(f"Comparative Analysis: Ligand Effects on Protein Interactions")
print(f"{'='*70}")

if len(all_system_data) < 2:
    print("ERROR: Need at least 2 systems for comparison")
else:
    # Combine all system means (now comparable because normalized)
    comparison_data = pd.DataFrame({
        name: data['overall_mean']
        for name, data in all_system_data.items()
    })
    
    # Get union of all interactions
    all_interactions = comparison_data.index
    print(f"\nTotal unique protein interactions found: {len(all_interactions)}")
    
    # Fill NaN with 0 for missing interactions
    comparison_data = comparison_data.fillna(0)
    
    # Filter interactions present in at least one system above threshold
    filtered_comparison = comparison_data[
        (comparison_data > interaction_threshold_low).any(axis=1)
    ]
    
    # Save comparison data
    comparison_file = os.path.join(output_dir, "ligand_comparison_all.csv")
    filtered_comparison.to_csv(comparison_file)
    print(f"\n✓ Saved comparison: {comparison_file}")
    print(f"  Interactions > {interaction_threshold_low}%: {len(filtered_comparison)}")
    
    # High-frequency interactions (>50% in at least one system)
    high_freq_comparison = comparison_data[
        (comparison_data > interaction_threshold_high).any(axis=1)
    ]
    
    high_freq_file = os.path.join(output_dir, "ligand_comparison_high_freq.csv")
    high_freq_comparison.to_csv(high_freq_file)
    print(f"✓ Saved high-frequency comparison: {high_freq_file}")
    print(f"  Interactions > {interaction_threshold_high}%: {len(high_freq_comparison)}")
    
    # =============================================================================
    # INTERACTION CATEGORIZATION
    # =============================================================================
    
    system_names = list(all_system_data.keys())
    
    # Categorize interactions
    common_interactions = comparison_data[
        (comparison_data[system_names[0]] > interaction_threshold_low) & 
        (comparison_data[system_names[1]] > interaction_threshold_low)
    ]
    
    unique_to_system1 = comparison_data[
        (comparison_data[system_names[0]] > interaction_threshold_low) & 
        (comparison_data[system_names[1]] <= interaction_threshold_low)
    ]
    
    unique_to_system2 = comparison_data[
        (comparison_data[system_names[0]] <= interaction_threshold_low) & 
        (comparison_data[system_names[1]] > interaction_threshold_low)
    ]
    
    print(f"\n{'='*70}")
    print(f"INTERACTION CATEGORIZATION")
    print(f"{'='*70}")
    print(f"Common interactions (>{interaction_threshold_low}% in both): {len(common_interactions)}")
    print(f"Unique to {system_names[0]}: {len(unique_to_system1)}")
    print(f"Unique to {system_names[1]}: {len(unique_to_system2)}")
    
    # Save categorized interactions
    common_interactions.to_csv(os.path.join(output_dir, "common_interactions.csv"))
    unique_to_system1.to_csv(os.path.join(output_dir, f"unique_to_{system_names[0]}.csv"))
    unique_to_system2.to_csv(os.path.join(output_dir, f"unique_to_{system_names[1]}.csv"))
    
    # =============================================================================
    # VISUALIZATIONS
    # =============================================================================
    
    print(f"\n{'='*70}")
    print(f"Generating Comparative Visualizations")
    print(f"{'='*70}")
    
    # 1. Comprehensive Heatmap (all significant interactions)
    if len(filtered_comparison) > 0:
        fig, ax = plt.subplots(figsize=(10, max(8, len(filtered_comparison) * 0.25)), dpi=150)
        
        sns.heatmap(
            filtered_comparison,
            annot=True,
            fmt='.1f',
            cmap='RdYlGn',
            center=50,
            vmin=0,
            vmax=100,
            cbar_kws={'label': 'Interaction Frequency (%)'},
            ax=ax,
            linewidths=0.5
        )
        
        ax.set_title('Ligand-Protein Interaction Comparison\n(Residue-Interaction Type)', 
                     fontsize=14, fontweight='bold')
        ax.set_xlabel('Ligand System', fontsize=12)
        ax.set_ylabel('Protein Interaction', fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        
        heatmap_file = os.path.join(output_dir, "comparison_heatmap_all.png")
        plt.savefig(heatmap_file, dpi=150, bbox_inches='tight')
        print(f"✓ Saved comprehensive heatmap: {heatmap_file}")
        plt.close()
    
    # 2. High-frequency interactions heatmap
    if len(high_freq_comparison) > 0:
        fig, ax = plt.subplots(figsize=(10, max(6, len(high_freq_comparison) * 0.3)), dpi=150)
        
        sns.heatmap(
            high_freq_comparison,
            annot=True,
            fmt='.1f',
            cmap='RdYlGn',
            center=50,
            vmin=0,
            vmax=100,
            cbar_kws={'label': 'Interaction Frequency (%)'},
            ax=ax,
            linewidths=0.5
        )
        
        ax.set_title('High-Frequency Interactions (>50%)\nAcross Ligand Systems', 
                     fontsize=14, fontweight='bold')
        ax.set_xlabel('Ligand System', fontsize=12)
        ax.set_ylabel('Protein Interaction', fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        
        heatmap_file = os.path.join(output_dir, "comparison_heatmap_high_freq.png")
        plt.savefig(heatmap_file, dpi=150, bbox_inches='tight')
        print(f"✓ Saved high-frequency heatmap: {heatmap_file}")
        plt.close()
    
    # 3. Scatter plot: direct comparison
    if len(all_system_data) == 2:
        fig, ax = plt.subplots(figsize=(10, 10), dpi=150)
        
        x = comparison_data[system_names[0]]
        y = comparison_data[system_names[1]]
        
        # Color points by category
        colors = []
        for idx in comparison_data.index:
            if idx in common_interactions.index:
                colors.append('blue')
            elif idx in unique_to_system1.index:
                colors.append('red')
            elif idx in unique_to_system2.index:
                colors.append('green')
            else:
                colors.append('gray')
        
        ax.scatter(x, y, alpha=0.6, c=colors, s=50)
        
        # Add diagonal line
        max_val = max(x.max(), y.max())
        ax.plot([0, max_val], [0, max_val], 'k--', alpha=0.3, label='Equal frequency')
        
        # Add threshold lines
        ax.axhline(y=interaction_threshold_low, color='orange', linestyle=':', alpha=0.5)
        ax.axvline(x=interaction_threshold_low, color='orange', linestyle=':', alpha=0.5)
        
        ax.set_xlabel(f'{system_names[0]} Interaction Frequency (%)', fontsize=12)
        ax.set_ylabel(f'{system_names[1]} Interaction Frequency (%)', fontsize=12)
        ax.set_title('Direct Ligand Comparison: Interaction Frequencies', 
                     fontsize=14, fontweight='bold')
        ax.grid(alpha=0.3)
        
        # Add legend
        from matplotlib.patches import Patch
        legend_elements = [
            Patch(facecolor='blue', label=f'Common (n={len(common_interactions)})'),
            Patch(facecolor='red', label=f'Unique to {system_names[0]} (n={len(unique_to_system1)})'),
            Patch(facecolor='green', label=f'Unique to {system_names[1]} (n={len(unique_to_system2)})'),
            Patch(facecolor='gray', label='Below threshold')
        ]
        ax.legend(handles=legend_elements, loc='upper left')
        
        plt.tight_layout()
        scatter_file = os.path.join(output_dir, "scatter_comparison.png")
        plt.savefig(scatter_file, dpi=150, bbox_inches='tight')
        print(f"✓ Saved scatter plot: {scatter_file}")
        plt.close()
    
    # 4. Difference analysis (for 2 systems)
    if len(all_system_data) == 2:
        diff = (comparison_data[system_names[0]] - comparison_data[system_names[1]])
        diff = diff[abs(diff) > 10].sort_values(ascending=False)
        
        if len(diff) > 0:
            fig, ax = plt.subplots(figsize=(12, max(8, len(diff) * 0.25)), dpi=150)
            
            colors = ['#2ca02c' if x > 0 else '#d62728' for x in diff.values]
            diff.plot(kind='barh', ax=ax, color=colors, width=0.8)
            
            ax.set_xlabel(f'Difference: {system_names[0]} - {system_names[1]} (%)', fontsize=12)
            ax.set_ylabel('Protein Interaction', fontsize=12)
            ax.set_title(f'Interaction Frequency Differences\nGreen: Favored by {system_names[0]}, Red: Favored by {system_names[1]}', 
                        fontsize=14, fontweight='bold')
            ax.axvline(x=0, color='black', linestyle='-', linewidth=1.5)
            ax.grid(axis='x', alpha=0.3)
            plt.tight_layout()
            
            diff_file = os.path.join(output_dir, "interaction_differences.png")
            plt.savefig(diff_file, dpi=150, bbox_inches='tight')
            print(f"✓ Saved difference plot: {diff_file}")
            plt.close()
            
            # Save difference data
            diff.to_csv(os.path.join(output_dir, "interaction_differences.csv"))
    
    # 5. Bar plot for top interactions in each category
    fig, axes = plt.subplots(1, 3, figsize=(20, 10), dpi=150)
    
    categories = [
        (common_interactions, 'Common Interactions', 'blue', axes[0]),
        (unique_to_system1, f'Unique to {system_names[0]}', 'red', axes[1]),
        (unique_to_system2, f'Unique to {system_names[1]}', 'green', axes[2])
    ]
    
    for data, title, color, ax in categories:
        if len(data) > 0:
            top_data = data.head(15)
            top_data.plot(kind='barh', ax=ax, width=0.8)
            ax.set_xlabel('Interaction Frequency (%)', fontsize=16)
            ax.set_title(title, fontsize=18, fontweight='bold')
            ax.legend(fontsize=14)
            ax.grid(axis='x', alpha=0.3)
            # Increase tick label sizes
            ax.tick_params(axis='both', labelsize=14)
        else:
            ax.text(0.5, 0.5, 'No interactions', ha='center', va='center', 
                    transform=ax.transAxes, fontsize=16)
            ax.set_title(title, fontsize=18, fontweight='bold')
    
    plt.tight_layout()
    category_file = os.path.join(output_dir, "interaction_categories.png")
    plt.savefig(category_file, dpi=150, bbox_inches='tight')
    print(f"✓ Saved category plot: {category_file}")
    plt.close()
    
    # 6. Generate barcode plots for each system
    for system_name, data in all_system_data.items():
        print(f"\n  Generating barcodes for {system_name}...")
        for res in data['results']:
            replica = res['replica']
            fp = res['fingerprint']
            
            try:
                fig = fp.plot_barcode(dpi=150, figsize=(15, 10))
                barcode_file = os.path.join(
                    data['output_dir'],
                    f"barcode_{system_name}_{replica}.png"
                )
                plt.savefig(barcode_file, dpi=150, bbox_inches='tight')
                plt.close()
                print(f"    ✓ {replica} barcode saved")
            except Exception as e:
                print(f"    ✗ Error creating barcode for {replica}: {e}")
    
    # =============================================================================
    # SUMMARY REPORT
    # =============================================================================
    
    print(f"\n{'='*70}")
    print(f"SUMMARY REPORT: Ligand Comparison")
    print(f"{'='*70}")
    
    for system_name, data in all_system_data.items():
        ligand_name = data['config']['ligand_resnames']
        print(f"\n{system_name} (Ligand: {ligand_name}):")
        print(f"  Replicas analyzed: {len(data['results'])}")
        if data['overall_mean'] is not None:
            total_interactions = len(data['overall_mean'][data['overall_mean'] > interaction_threshold_low])
            print(f"  Total significant interactions (>{interaction_threshold_low}%): {total_interactions}")
            top_5 = data['overall_mean'].head(5)
            print(f"  Top 5 interactions:")
            for interaction, freq in top_5.items():
                print(f"    - {interaction}: {freq:.1f}%")
    
    print(f"\n{'='*70}")
    print(f"Comparative Summary:")
    print(f"{'='*70}")
    print(f"Common interactions: {len(common_interactions)}")
    print(f"Unique to {system_names[0]}: {len(unique_to_system1)}")
    print(f"Unique to {system_names[1]}: {len(unique_to_system2)}")
    
    print(f"\n{'='*70}")
    print(f"Analysis Complete!")
    print(f"{'='*70}")
    print(f"\nAll outputs saved to: {output_dir}/")
    print(f"\nKey files:")
    print(f"  - ligand_comparison_all.csv: All significant interactions")
    print(f"  - ligand_comparison_high_freq.csv: High-frequency interactions")
    print(f"  - common_interactions.csv: Shared by both ligands")
    print(f"  - unique_to_*.csv: Ligand-specific interactions")
    print(f"  - comparison_heatmap_*.png: Visual comparisons")
    print(f"  - scatter_comparison.png: Direct frequency comparison")
    print(f"  - interaction_differences.png: Difference plot")
    print(f"  - interaction_categories.png: Categorized interactions")


######################################################################
#  Multi-Ligand Interaction Comparison Analysis
#  Comparing: GGPP, FSPP
######################################################################

Loading System: GGPP - PSY with GGPP ligand
  Loading MD: /media/dlbox/8TB/New_simulation/PQR/3w7f/with_GGPP/run//MD/MMPBSA/center2x.nc
    ✓ Loaded 10000 frames
  Loading MD2: /media/dlbox/8TB/New_simulation/PQR/3w7f/with_GGPP/run//MD2/MMPBSA/center2x.nc
    ✓ Loaded 10000 frames
  Loading MD3: /media/dlbox/8TB/New_simulation/PQR/3w7f/with_GGPP/run//MD3/MMPBSA/center2x.nc
    ✓ Loaded 10000 frames
  [MD] Ligand: 124 atoms, Protein: 132 residues
  [MD2] Ligand: 124 atoms, Protein: 139 residues
  [MD3] Ligand: 124 atoms, Protein: 134 residues

Running Fingerprint Analysis: GGPP

  [MD] Analyzing every 50th frame...


  0%|          | 0/200 [00:00<?, ?it/s]

    ✓ Analyzed 200 frames
    ✓ Found 69 unique protein interactions

  [MD2] Analyzing every 50th frame...


  0%|          | 0/200 [00:00<?, ?it/s]

    ✓ Analyzed 200 frames
    ✓ Found 70 unique protein interactions

  [MD3] Analyzing every 50th frame...


...

    ✓ Analyzed 200 frames
    ✓ Found 67 unique protein interactions

  [MD2] Analyzing every 50th frame...


  0%|          | 0/200 [00:00<?, ?it/s]

    ✓ Analyzed 200 frames
    ✓ Found 68 unique protein interactions

  [MD3] Analyzing every 50th frame...


  0%|          | 0/200 [00:00<?, ?it/s]

    ✓ Analyzed 200 frames
    ✓ Found 69 unique protein interactions

Comparative Analysis: Ligand Effects on Protein Interactions

Total unique protein interactions found: 88

✓ Saved comparison: 2ligand_comparison_analysis_CrtM/ligand_comparison_all.csv
  Interactions > 50.0%: 49
✓ Saved high-frequency comparison: 2ligand_comparison_analysis_CrtM/ligand_comparison_high_freq.csv
  Interactions > 80.0%: 41

INTERACTION CATEGORIZATION
Common interactions (>50.0% in both): 33
Unique to GGPP: 7
Unique to FSPP: 9

Generating Comparative Visualizations
✓ Saved comprehensive heatmap: 2ligand_comparison_analysis_CrtM/comparison_heatmap_all.png
✓ Saved high-frequency heatmap: 2ligand_comparison_analysis_CrtM/comparison_heatmap_high_freq.png
✓ Saved scatter plot: 2ligand_comparison_analysis_CrtM/scatter_comparison.png
✓ Saved difference plot: 2ligand_comparison_analysis_CrtM/interaction_differences.png
✓ Saved category plot: 2ligand_comparison_analysis_CrtM/interaction_categories.png

  Generat

In [6]:
import os
import MDAnalysis as mda
import prolif as plf
import pandas as pd
import seaborn as sns
import numpy as np
from matplotlib import pyplot as plt
from rdkit import DataStructs

# =============================================================================
# CONFIGURATION - Define all your systems here
# =============================================================================

systems = {
    "GGPP": {
        "root_dir": "/media/dlbox/6TB/NEW_PSY/with_FPP/run/",
        "replica_folders": ["MD", "MD2", "MD3", "MDxx", "MD2xx"],
        "traj_filename": "MMPBSA/center2.nc",
        "ligand_resnames": "LG*",  # Main ligand to analyze (excluding MG)
        "cofactor_resnames": "MG",  # Optional cofactor
        "description": "PSY with GGPP ligand"
    },
    "FSPP": {
        "root_dir": "/media/dlbox/6TB/NEW_PSY/run/",
        "replica_folders": ["MD", "MD2", "MD3", "MDxxx", "MD2xxx"],
        "traj_filename": "MMPBSA/center2.nc",
        "ligand_resnames": "LG*",  # Main ligand to analyze (excluding MG)
        "cofactor_resnames": "MG",  # Optional cofactor
        "description": "PSY with FSPP ligand"
    }
}

# Analysis parameters
frame_step = 50  # Analyze every Nth frame
distance_cutoff = 10.0  # Å around ligand for protein selection
interaction_threshold_low = 50.0  # % for filtering
interaction_threshold_high = 80.0  # % for high-frequency interactions

# Output directory
output_dir = "ligand_comparison_analysis_NEW_PSY"
os.makedirs(output_dir, exist_ok=True)

# =============================================================================
# FUNCTIONS
# =============================================================================

def load_system(system_name, system_config):
    """Load all replicas for a given system."""
    print(f"\n{'='*70}")
    print(f"Loading System: {system_name} - {system_config['description']}")
    print(f"{'='*70}")
    
    universes = []
    root_dir = system_config['root_dir']
    
    for replica in system_config['replica_folders']:
        top_file = f"{root_dir}/{replica}/solvated.prmtop"
        traj_file = f"{root_dir}/{replica}/{system_config['traj_filename']}"
        
        if not os.path.exists(top_file):
            print(f"WARNING: Topology file not found: {top_file}")
            continue
        if not os.path.exists(traj_file):
            print(f"WARNING: Trajectory file not found: {traj_file}")
            continue
            
        print(f"  Loading {replica}: {traj_file}")
        try:
            u = mda.Universe(top_file, traj_file)
            universes.append({'universe': u, 'replica': replica})
            print(f"    ✓ Loaded {len(u.trajectory)} frames")
        except Exception as e:
            print(f"    ✗ Error loading {replica}: {e}")
            
    return universes

def create_selections(universes, system_config, distance_cutoff):
    """Create ligand and protein selections for all replicas."""
    selections = []
    
    ligand_resnames = system_config['ligand_resnames']
    cofactor_resnames = system_config.get('cofactor_resnames', '')
    
    for entry in universes:
        u = entry['universe']
        replica = entry['replica']
        
        try:
            # Select main ligand only
            ligand_sel = u.select_atoms(f"resname {ligand_resnames}")
            
            # Select all ligands including cofactor for display (optional)
            if cofactor_resnames:
                all_ligands = u.select_atoms(f"resname {ligand_resnames} {cofactor_resnames}")
            else:
                all_ligands = ligand_sel
            
            # Select protein within distance cutoff of main ligand
            protein_sel = u.select_atoms(
                f"protein and byres around {distance_cutoff} group ligand",
                ligand=ligand_sel
            )
            
            # Create ProLIF molecules
            ligand_mol = plf.Molecule.from_mda(ligand_sel)
            protein_mol = plf.Molecule.from_mda(protein_sel)
            
            selections.append({
                'replica': replica,
                'universe': u,
                'ligand_sel': ligand_sel,
                'protein_sel': protein_sel,
                'ligand_mol': ligand_mol,
                'protein_mol': protein_mol
            })
            
            print(f"  [{replica}] Ligand: {len(ligand_sel)} atoms, "
                  f"Protein: {len(protein_sel.residues)} residues")
            
        except Exception as e:
            print(f"  [{replica}] Error creating selections: {e}")
            
    return selections

def normalize_interaction_keys(df):
    """
    Normalize interaction keys to remove ligand residue information.
    ProLIF uses MultiIndex columns with levels: ligand, protein, interaction
    We need to drop the ligand level and group by (protein, interaction)
    """
    # Check if columns are MultiIndex
    if isinstance(df.columns, pd.MultiIndex):
        # Get the number of levels
        n_levels = df.columns.nlevels
        
        if n_levels == 3:
            # Format: (ligand, protein, interaction)
            # Drop ligand level (level 0), keep protein and interaction
            df_normalized = df.copy()
            
            # Create new column labels without ligand info
            new_cols = [(col[1], col[2]) for col in df.columns]
            df_normalized.columns = pd.MultiIndex.from_tuples(new_cols, names=['protein', 'interaction'])
            
            # Group by the new column names (sum if multiple ligands interact with same residue)
            df_normalized = df_normalized.T.groupby(level=[0, 1]).sum().T
            
        else:
            print(f"  Warning: Unexpected MultiIndex levels: {n_levels}")
            df_normalized = df
    else:
        # If not MultiIndex, try tuple-based normalization
        new_columns = {}
        for col in df.columns:
            if isinstance(col, tuple) and len(col) == 3:
                # col format: (ligand_res, protein_res, interaction_type)
                ligand_res, protein_res, interaction_type = col
                new_key = (protein_res, interaction_type)
                new_columns[col] = new_key
            else:
                new_columns[col] = col
        
        df_normalized = df.rename(columns=new_columns)
        
        # If duplicate columns exist, sum them
        if df_normalized.columns.duplicated().any():
            df_normalized = df_normalized.T.groupby(level=0).sum().T
    
    return df_normalized

def run_fingerprint_analysis(selections, system_name, frame_step, output_dir):
    """Run interaction fingerprint analysis."""
    print(f"\n{'='*70}")
    print(f"Running Fingerprint Analysis: {system_name}")
    print(f"{'='*70}")
    
    results = []
    
    for sel in selections:
        replica = sel['replica']
        print(f"\n  [{replica}] Analyzing every {frame_step}th frame...")
        
        try:
            # Initialize and run fingerprint
            fp = plf.Fingerprint()
            fp.run(
                sel['universe'].trajectory[0::frame_step],
                sel['ligand_sel'],
                sel['protein_sel']
            )
            
            # Save fingerprint
            pkl_file = os.path.join(output_dir, f"fp_{system_name}_{replica}.pkl")
            fp.to_pickle(pkl_file)
            
            # Convert to dataframe and normalize
            df = fp.to_dataframe()
            df_normalized = normalize_interaction_keys(df)
            
            results.append({
                'replica': replica,
                'fingerprint': fp,
                'dataframe': df,
                'dataframe_normalized': df_normalized,
                'pkl_file': pkl_file
            })
            
            print(f"    ✓ Analyzed {len(df)} frames")
            print(f"    ✓ Found {len(df_normalized.columns)} unique protein interactions")
            
        except Exception as e:
            print(f"    ✗ Error: {e}")
            import traceback
            traceback.print_exc()
            
    return results

def combine_replica_data(results, system_name):
    """Combine interaction data across replicas using normalized data."""
    if not results:
        return None, None, None
    
    # Calculate mean interactions per replica (as percentage) using normalized data
    interaction_means = []
    all_normalized_dfs = []
    
    for res in results:
        df_norm = res['dataframe_normalized']
        all_normalized_dfs.append(df_norm)
        mean_vals = df_norm.mean().sort_values(ascending=False) * 100
        interaction_means.append(mean_vals.to_frame(name=res['replica']).T)
    
    # Combine all replicas
    combined_data = pd.concat(interaction_means, ignore_index=False)
    
    # Calculate overall average across replicas
    overall_mean = combined_data.mean().sort_values(ascending=False)
    
    # Concatenate all normalized dataframes for full dataset
    all_frames = pd.concat(all_normalized_dfs, ignore_index=True)
    
    return combined_data, overall_mean, all_frames

# =============================================================================
# MAIN ANALYSIS PIPELINE
# =============================================================================

print(f"\n{'#'*70}")
print(f"#  Multi-Ligand Interaction Comparison Analysis")
print(f"#  Comparing: {', '.join(systems.keys())}")
print(f"{'#'*70}")

# Store all system data
all_system_data = {}

# Process each system
for system_name, system_config in systems.items():
    system_output = os.path.join(output_dir, system_name)
    os.makedirs(system_output, exist_ok=True)
    
    # Load trajectories
    universes = load_system(system_name, system_config)
    if not universes:
        print(f"ERROR: No valid trajectories loaded for {system_name}")
        continue
    
    # Create selections
    selections = create_selections(
        universes,
        system_config,
        distance_cutoff
    )
    if not selections:
        print(f"ERROR: No valid selections for {system_name}")
        continue
    
    # Run fingerprint analysis
    results = run_fingerprint_analysis(
        selections,
        system_name,
        frame_step,
        system_output
    )
    if not results:
        print(f"ERROR: No fingerprint results for {system_name}")
        continue
    
    # Combine replica data (using normalized interactions)
    combined_data, overall_mean, all_frames = combine_replica_data(results, system_name)
    
    # Store system data
    all_system_data[system_name] = {
        'selections': selections,
        'results': results,
        'combined_data': combined_data,
        'overall_mean': overall_mean,
        'all_frames': all_frames,
        'output_dir': system_output,
        'config': system_config
    }
    
    # Save system-specific data
    if combined_data is not None:
        combined_data.T.to_csv(
            os.path.join(system_output, f"{system_name}_all_interactions.csv")
        )
        overall_mean.to_csv(
            os.path.join(system_output, f"{system_name}_mean_interactions.csv")
        )

# =============================================================================
# COMPARATIVE ANALYSIS
# =============================================================================

print(f"\n{'='*70}")
print(f"Comparative Analysis: Ligand Effects on Protein Interactions")
print(f"{'='*70}")

if len(all_system_data) < 2:
    print("ERROR: Need at least 2 systems for comparison")
else:
    # Combine all system means (now comparable because normalized)
    comparison_data = pd.DataFrame({
        name: data['overall_mean']
        for name, data in all_system_data.items()
    })
    
    # Get union of all interactions
    all_interactions = comparison_data.index
    print(f"\nTotal unique protein interactions found: {len(all_interactions)}")
    
    # Fill NaN with 0 for missing interactions
    comparison_data = comparison_data.fillna(0)
    
    # Filter interactions present in at least one system above threshold
    filtered_comparison = comparison_data[
        (comparison_data > interaction_threshold_low).any(axis=1)
    ]
    
    # Save comparison data
    comparison_file = os.path.join(output_dir, "ligand_comparison_all.csv")
    filtered_comparison.to_csv(comparison_file)
    print(f"\n✓ Saved comparison: {comparison_file}")
    print(f"  Interactions > {interaction_threshold_low}%: {len(filtered_comparison)}")
    
    # High-frequency interactions (>50% in at least one system)
    high_freq_comparison = comparison_data[
        (comparison_data > interaction_threshold_high).any(axis=1)
    ]
    
    high_freq_file = os.path.join(output_dir, "ligand_comparison_high_freq.csv")
    high_freq_comparison.to_csv(high_freq_file)
    print(f"✓ Saved high-frequency comparison: {high_freq_file}")
    print(f"  Interactions > {interaction_threshold_high}%: {len(high_freq_comparison)}")
    
    # =============================================================================
    # INTERACTION CATEGORIZATION
    # =============================================================================
    
    system_names = list(all_system_data.keys())
    
    # Categorize interactions
    common_interactions = comparison_data[
        (comparison_data[system_names[0]] > interaction_threshold_low) & 
        (comparison_data[system_names[1]] > interaction_threshold_low)
    ]
    
    unique_to_system1 = comparison_data[
        (comparison_data[system_names[0]] > interaction_threshold_low) & 
        (comparison_data[system_names[1]] <= interaction_threshold_low)
    ]
    
    unique_to_system2 = comparison_data[
        (comparison_data[system_names[0]] <= interaction_threshold_low) & 
        (comparison_data[system_names[1]] > interaction_threshold_low)
    ]
    
    print(f"\n{'='*70}")
    print(f"INTERACTION CATEGORIZATION")
    print(f"{'='*70}")
    print(f"Common interactions (>{interaction_threshold_low}% in both): {len(common_interactions)}")
    print(f"Unique to {system_names[0]}: {len(unique_to_system1)}")
    print(f"Unique to {system_names[1]}: {len(unique_to_system2)}")
    
    # Save categorized interactions
    common_interactions.to_csv(os.path.join(output_dir, "common_interactions.csv"))
    unique_to_system1.to_csv(os.path.join(output_dir, f"unique_to_{system_names[0]}.csv"))
    unique_to_system2.to_csv(os.path.join(output_dir, f"unique_to_{system_names[1]}.csv"))
    
    # =============================================================================
    # VISUALIZATIONS
    # =============================================================================
    
    print(f"\n{'='*70}")
    print(f"Generating Comparative Visualizations")
    print(f"{'='*70}")
    
    # 1. Comprehensive Heatmap (all significant interactions)
    if len(filtered_comparison) > 0:
        fig, ax = plt.subplots(figsize=(10, max(8, len(filtered_comparison) * 0.25)), dpi=150)
        
        sns.heatmap(
            filtered_comparison,
            annot=True,
            fmt='.1f',
            cmap='RdYlGn',
            center=50,
            vmin=0,
            vmax=100,
            cbar_kws={'label': 'Interaction Frequency (%)'},
            ax=ax,
            linewidths=0.5
        )
        
        ax.set_title('Ligand-Protein Interaction Comparison\n(Residue-Interaction Type)', 
                     fontsize=14, fontweight='bold')
        ax.set_xlabel('Ligand System', fontsize=12)
        ax.set_ylabel('Protein Interaction', fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        
        heatmap_file = os.path.join(output_dir, "comparison_heatmap_all.png")
        plt.savefig(heatmap_file, dpi=150, bbox_inches='tight')
        print(f"✓ Saved comprehensive heatmap: {heatmap_file}")
        plt.close()
    
    # 2. High-frequency interactions heatmap
    if len(high_freq_comparison) > 0:
        fig, ax = plt.subplots(figsize=(10, max(6, len(high_freq_comparison) * 0.3)), dpi=150)
        
        sns.heatmap(
            high_freq_comparison,
            annot=True,
            fmt='.1f',
            cmap='RdYlGn',
            center=50,
            vmin=0,
            vmax=100,
            cbar_kws={'label': 'Interaction Frequency (%)'},
            ax=ax,
            linewidths=0.5
        )
        
        ax.set_title('High-Frequency Interactions (>50%)\nAcross Ligand Systems', 
                     fontsize=14, fontweight='bold')
        ax.set_xlabel('Ligand System', fontsize=12)
        ax.set_ylabel('Protein Interaction', fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        
        heatmap_file = os.path.join(output_dir, "comparison_heatmap_high_freq.png")
        plt.savefig(heatmap_file, dpi=150, bbox_inches='tight')
        print(f"✓ Saved high-frequency heatmap: {heatmap_file}")
        plt.close()
    
    # 3. Scatter plot: direct comparison
    if len(all_system_data) == 2:
        fig, ax = plt.subplots(figsize=(10, 10), dpi=150)
        
        x = comparison_data[system_names[0]]
        y = comparison_data[system_names[1]]
        
        # Color points by category
        colors = []
        for idx in comparison_data.index:
            if idx in common_interactions.index:
                colors.append('blue')
            elif idx in unique_to_system1.index:
                colors.append('red')
            elif idx in unique_to_system2.index:
                colors.append('green')
            else:
                colors.append('gray')
        
        ax.scatter(x, y, alpha=0.6, c=colors, s=50)
        
        # Add diagonal line
        max_val = max(x.max(), y.max())
        ax.plot([0, max_val], [0, max_val], 'k--', alpha=0.3, label='Equal frequency')
        
        # Add threshold lines
        ax.axhline(y=interaction_threshold_low, color='orange', linestyle=':', alpha=0.5)
        ax.axvline(x=interaction_threshold_low, color='orange', linestyle=':', alpha=0.5)
        
        ax.set_xlabel(f'{system_names[0]} Interaction Frequency (%)', fontsize=12)
        ax.set_ylabel(f'{system_names[1]} Interaction Frequency (%)', fontsize=12)
        ax.set_title('Direct Ligand Comparison: Interaction Frequencies', 
                     fontsize=14, fontweight='bold')
        ax.grid(alpha=0.3)
        
        # Add legend
        from matplotlib.patches import Patch
        legend_elements = [
            Patch(facecolor='blue', label=f'Common (n={len(common_interactions)})'),
            Patch(facecolor='red', label=f'Unique to {system_names[0]} (n={len(unique_to_system1)})'),
            Patch(facecolor='green', label=f'Unique to {system_names[1]} (n={len(unique_to_system2)})'),
            Patch(facecolor='gray', label='Below threshold')
        ]
        ax.legend(handles=legend_elements, loc='upper left')
        
        plt.tight_layout()
        scatter_file = os.path.join(output_dir, "scatter_comparison.png")
        plt.savefig(scatter_file, dpi=150, bbox_inches='tight')
        print(f"✓ Saved scatter plot: {scatter_file}")
        plt.close()
    
    # 4. Difference analysis (for 2 systems)
    if len(all_system_data) == 2:
        diff = (comparison_data[system_names[0]] - comparison_data[system_names[1]])
        diff = diff[abs(diff) > 10].sort_values(ascending=False)
        
        if len(diff) > 0:
            fig, ax = plt.subplots(figsize=(12, max(8, len(diff) * 0.25)), dpi=150)
            
            colors = ['#2ca02c' if x > 0 else '#d62728' for x in diff.values]
            diff.plot(kind='barh', ax=ax, color=colors, width=0.8)
            
            ax.set_xlabel(f'Difference: {system_names[0]} - {system_names[1]} (%)', fontsize=12)
            ax.set_ylabel('Protein Interaction', fontsize=12)
            ax.set_title(f'Interaction Frequency Differences\nGreen: Favored by {system_names[0]}, Red: Favored by {system_names[1]}', 
                        fontsize=14, fontweight='bold')
            ax.axvline(x=0, color='black', linestyle='-', linewidth=1.5)
            ax.grid(axis='x', alpha=0.3)
            plt.tight_layout()
            
            diff_file = os.path.join(output_dir, "interaction_differences.png")
            plt.savefig(diff_file, dpi=150, bbox_inches='tight')
            print(f"✓ Saved difference plot: {diff_file}")
            plt.close()
            
            # Save difference data
            diff.to_csv(os.path.join(output_dir, "interaction_differences.csv"))
    
    # 5. Bar plot for top interactions in each category
    fig, axes = plt.subplots(1, 3, figsize=(20, 10), dpi=150)
    
    categories = [
        (common_interactions, 'Common Interactions', 'blue', axes[0]),
        (unique_to_system1, f'Unique to {system_names[0]}', 'red', axes[1]),
        (unique_to_system2, f'Unique to {system_names[1]}', 'green', axes[2])
    ]
    
    for data, title, color, ax in categories:
        if len(data) > 0:
            top_data = data.head(15)
            top_data.plot(kind='barh', ax=ax, width=0.8)
            ax.set_xlabel('Interaction Frequency (%)', fontsize=16)
            ax.set_title(title, fontsize=18, fontweight='bold')
            ax.legend(fontsize=14)
            ax.grid(axis='x', alpha=0.3)
            # Increase tick label sizes
            ax.tick_params(axis='both', labelsize=14)
        else:
            ax.text(0.5, 0.5, 'No interactions', ha='center', va='center', 
                    transform=ax.transAxes, fontsize=16)
            ax.set_title(title, fontsize=18, fontweight='bold')
    
    plt.tight_layout()
    category_file = os.path.join(output_dir, "interaction_categories.png")
    plt.savefig(category_file, dpi=150, bbox_inches='tight')
    print(f"✓ Saved category plot: {category_file}")
    plt.close()
    
    # 6. Generate barcode plots for each system
    for system_name, data in all_system_data.items():
        print(f"\n  Generating barcodes for {system_name}...")
        for res in data['results']:
            replica = res['replica']
            fp = res['fingerprint']
            
            try:
                fig = fp.plot_barcode(dpi=150, figsize=(15, 10))
                barcode_file = os.path.join(
                    data['output_dir'],
                    f"barcode_{system_name}_{replica}.png"
                )
                plt.savefig(barcode_file, dpi=150, bbox_inches='tight')
                plt.close()
                print(f"    ✓ {replica} barcode saved")
            except Exception as e:
                print(f"    ✗ Error creating barcode for {replica}: {e}")
    
    # =============================================================================
    # SUMMARY REPORT
    # =============================================================================
    
    print(f"\n{'='*70}")
    print(f"SUMMARY REPORT: Ligand Comparison")
    print(f"{'='*70}")
    
    for system_name, data in all_system_data.items():
        ligand_name = data['config']['ligand_resnames']
        print(f"\n{system_name} (Ligand: {ligand_name}):")
        print(f"  Replicas analyzed: {len(data['results'])}")
        if data['overall_mean'] is not None:
            total_interactions = len(data['overall_mean'][data['overall_mean'] > interaction_threshold_low])
            print(f"  Total significant interactions (>{interaction_threshold_low}%): {total_interactions}")
            top_5 = data['overall_mean'].head(5)
            print(f"  Top 5 interactions:")
            for interaction, freq in top_5.items():
                print(f"    - {interaction}: {freq:.1f}%")
    
    print(f"\n{'='*70}")
    print(f"Comparative Summary:")
    print(f"{'='*70}")
    print(f"Common interactions: {len(common_interactions)}")
    print(f"Unique to {system_names[0]}: {len(unique_to_system1)}")
    print(f"Unique to {system_names[1]}: {len(unique_to_system2)}")
    
    print(f"\n{'='*70}")
    print(f"Analysis Complete!")
    print(f"{'='*70}")
    print(f"\nAll outputs saved to: {output_dir}/")
    print(f"\nKey files:")
    print(f"  - ligand_comparison_all.csv: All significant interactions")
    print(f"  - ligand_comparison_high_freq.csv: High-frequency interactions")
    print(f"  - common_interactions.csv: Shared by both ligands")
    print(f"  - unique_to_*.csv: Ligand-specific interactions")
    print(f"  - comparison_heatmap_*.png: Visual comparisons")
    print(f"  - scatter_comparison.png: Direct frequency comparison")
    print(f"  - interaction_differences.png: Difference plot")
    print(f"  - interaction_categories.png: Categorized interactions")


######################################################################
#  Multi-Ligand Interaction Comparison Analysis
#  Comparing: GGPP, FSPP
######################################################################

Loading System: GGPP - PSY with GGPP ligand
  Loading MD: /media/dlbox/6TB/NEW_PSY/with_FPP/run//MD/MMPBSA/center2.nc
    ✓ Loaded 10000 frames
  Loading MD2: /media/dlbox/6TB/NEW_PSY/with_FPP/run//MD2/MMPBSA/center2.nc
    ✓ Loaded 10000 frames
  Loading MD3: /media/dlbox/6TB/NEW_PSY/with_FPP/run//MD3/MMPBSA/center2.nc
    ✓ Loaded 10000 frames
  Loading MDxx: /media/dlbox/6TB/NEW_PSY/with_FPP/run//MDxx/MMPBSA/center2.nc
    ✓ Loaded 10000 frames
  Loading MD2xx: /media/dlbox/6TB/NEW_PSY/with_FPP/run//MD2xx/MMPBSA/center2.nc
    ✓ Loaded 10000 frames
  [MD] Ligand: 98 atoms, Protein: 135 residues
  [MD2] Ligand: 98 atoms, Protein: 141 residues
  [MD3] Ligand: 98 atoms, Protein: 146 residues
  [MDxx] Ligand: 98 atoms, Protein: 143 residues
  [MD2xx] Ligand: 98 atoms, Prote

  0%|          | 0/200 [00:00<?, ?it/s]

    ✓ Analyzed 200 frames
    ✓ Found 69 unique protein interactions

  [MD2] Analyzing every 50th frame...


  0%|          | 0/200 [00:00<?, ?it/s]

    ✓ Analyzed 200 frames
    ✓ Found 66 unique protein interactions

  [MD3] Analyzing every 50th frame...


...

    ✓ Analyzed 200 frames
    ✓ Found 84 unique protein interactions

  [MDxxx] Analyzing every 50th frame...


  0%|          | 0/200 [00:00<?, ?it/s]

    ✓ Analyzed 200 frames
    ✓ Found 94 unique protein interactions

  [MD2xxx] Analyzing every 50th frame...


  0%|          | 0/200 [00:00<?, ?it/s]

    ✓ Analyzed 200 frames
    ✓ Found 89 unique protein interactions

Comparative Analysis: Ligand Effects on Protein Interactions

Total unique protein interactions found: 117

✓ Saved comparison: ligand_comparison_analysis_NEW_PSY/ligand_comparison_all.csv
  Interactions > 50.0%: 58
✓ Saved high-frequency comparison: ligand_comparison_analysis_NEW_PSY/ligand_comparison_high_freq.csv
  Interactions > 80.0%: 36

INTERACTION CATEGORIZATION
Common interactions (>50.0% in both): 32
Unique to GGPP: 14
Unique to FSPP: 12

Generating Comparative Visualizations
✓ Saved comprehensive heatmap: ligand_comparison_analysis_NEW_PSY/comparison_heatmap_all.png
✓ Saved high-frequency heatmap: ligand_comparison_analysis_NEW_PSY/comparison_heatmap_high_freq.png
✓ Saved scatter plot: ligand_comparison_analysis_NEW_PSY/scatter_comparison.png
✓ Saved difference plot: ligand_comparison_analysis_NEW_PSY/interaction_differences.png
✓ Saved category plot: ligand_comparison_analysis_NEW_PSY/interaction_categori

In [ ]:
/media/dlbox/6TB/NEW_PSY/with_FPP/run/